In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import random
import copy
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import random
import copy
import gc
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

DIR_BASE = 'C:/Proyecto_Embeddings'

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Entorno listo. Usando: {device}")

In [ ]:
COLS_X = 772
DTYPE_X = np.float16

class OracionesDataset(Dataset):
    def __init__(self, dir_base, split='train'):
        self.split = split

        path_X = os.path.join(dir_base, f'dataset_{split}_completo.dat')
        path_n_sent = os.path.join(dir_base, f'n_sentence_{split}.npy')

        prefix = 'y_train' if split == 'train' else 'y_eval'
        path_y_cap = os.path.join(dir_base, f'{prefix}_cap.npy')
        path_y_ini = os.path.join(dir_base, f'{prefix}_ini.npy')
        path_y_fin = os.path.join(dir_base, f'{prefix}_fin.npy')

        # Conectar a X (Memmap)
        filesize = os.path.getsize(path_X)
        n_rows = filesize // (COLS_X * np.dtype(DTYPE_X).itemsize)
        self.X = np.memmap(path_X, dtype=DTYPE_X, mode='r', shape=(n_rows, COLS_X))

        # Cargar Mapa y Targets (RAM)
        print(f"[{split.upper()}] Cargando índices...")
        self.n_sentence = np.load(path_n_sent)
        self.y_cap = np.load(path_y_cap)
        self.y_ini = np.load(path_y_ini)
        self.y_fin = np.load(path_y_fin)

        # Indexar
        _, start_indices, lengths = np.unique(self.n_sentence, return_index=True, return_counts=True)
        self.starts = start_indices
        self.lengths = lengths
        self.num_sentences = len(start_indices)

    def __len__(self):
        return self.num_sentences

    def __getitem__(self, idx):
        start = self.starts[idx]
        length = self.lengths[idx]
        end = start + length

        # Features
        x_seq = torch.tensor(self.X[start:end], dtype=torch.float32)

        # Construir Target
        target_list = []
        raw_c = self.y_cap[start:end]
        raw_i = self.y_ini[start:end]
        raw_f = self.y_fin[start:end]

        for i in range(length):
            # Binario Inicial
            v_ini = [1.0] if (str(raw_i[i]) not in ['', 'nan']) else [0.0]

            # One-Hot Final
            v_fin = [0.0]*4
            opts_f = ['', '.', ',', '?']
            val_f = str(raw_f[i])
            if val_f in opts_f: v_fin[opts_f.index(val_f)] = 1.0
            else: v_fin[0] = 1.0

            # One-Hot Capitalización
            v_cap = [0.0]*4
            val_c = int(raw_c[i])
            if 0 <= val_c <= 3: v_cap[val_c] = 1.0

            target_list.append(v_ini + v_fin + v_cap)

        y_seq = torch.tensor(target_list, dtype=torch.float32)
        return x_seq, y_seq

def collate_pad(batch):
    xx, yy = zip(*batch)
    x_pad = pad_sequence(xx, batch_first=True, padding_value=0)
    y_pad = pad_sequence(yy, batch_first=True, padding_value=0)
    return x_pad, y_pad

print(" Clase Dataset definida.")

In [ ]:
class NuestraLSTM_Uni(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_prob):
        super(NuestraLSTM_Uni, self).__init__()

        # LSTM Unidireccional
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout_prob if num_layers > 1 else 0,
            bidirectional=False
        )

        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out)
        logits = self.fc(out)
        return logits

class MixedLoss(nn.Module):
    def __init__(self, w_bin, w_soft1, w_soft2):
        super(MixedLoss, self).__init__()
        self.loss_bin = nn.BCEWithLogitsLoss(reduction='none')
        self.loss_mc = nn.CrossEntropyLoss(reduction='none')
        self.weights = (w_bin, w_soft1, w_soft2)

    def forward(self, logits, targets):
        # Máscara (ignorar padding)
        mask = (targets.sum(dim=-1) != 0).float()

        # Separar targets
        t_bin = targets[..., 0]
        t_s1 = targets[..., 1:5].argmax(dim=-1) # Puntuacion
        t_s2 = targets[..., 5:9].argmax(dim=-1) # Capitalizacion

        # Losses
        loss_b = self.loss_bin(logits[..., 0], t_bin) * mask
        loss_m1 = self.loss_mc(logits[..., 1:5].permute(0,2,1), t_s1) * mask
        loss_m2 = self.loss_mc(logits[..., 5:9].permute(0,2,1), t_s2) * mask

        N = mask.sum() + 1e-8
        return (self.weights[0]*loss_b.sum()/N +
                self.weights[1]*loss_m1.sum()/N +
                self.weights[2]*loss_m2.sum()/N)

print(" Modelo y Loss definidos.")

In [ ]:
train_ds = OracionesDataset(DIR_BASE, split='train')
eval_ds  = OracionesDataset(DIR_BASE, split='eval')

BATCH_SIZE = 32

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_pad,
    pin_memory=True,  # acelera transferencia a GPU
    num_workers=0
)

eval_loader = DataLoader(
    eval_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_pad,
    pin_memory=True,
    num_workers=0
)

print(f"DataLoaders listos con Batch Size: {BATCH_SIZE}")

# grilla de Hiperparámetros
param_grid = {
    'hidden_size': [128, 256, 512],
    'num_layers': [1, 2, 3],
    'dropout': [0.3, 0.4, 0.5],
    'lr': [0.001, 0.0005],

#Pesos Loss: (Binaria, Puntuacion, Capitalizacion)
    'loss_weights': [
        (1.0, 1.0, 1.0),
        (1.0, 2.0, 1.0),
        (1.0, 3.0, 1.0)
    ]
}

NUM_TRIALS = 5
EPOCHS_PER_TRIAL = 10
PATIENCE = 2

print(f"Listo para iniciar Random Search: {NUM_TRIALS} intentos.")

In [ ]:
from torch.cuda.amp import autocast, GradScaler
import gc

best_global_f1 = 0.0
best_global_params = {}
best_model_state = None

scaler = GradScaler()

print(f"INICIANDO BÚSQUEDA EN: {device}")

for trial in range(NUM_TRIALS):
    # Elegir parámetros
    params = {k: random.choice(v) for k, v in param_grid.items()}
    w_bin, w_s1, w_s2 = params['loss_weights']

    print(f"\n{'='*60}")
    print(f"TRIAL {trial+1}/{NUM_TRIALS} | Params: {params}")

    # Modelo
    model = NuestraLSTM_Uni(
        input_size=772,
        hidden_size=params['hidden_size'],
        num_layers=params['num_layers'],
        output_size=9,
        dropout_prob=params['dropout']
    ).to(device)

    criterion = MixedLoss(w_bin, w_s1, w_s2)
    optimizer = optim.Adam(model.parameters(), lr=params['lr'])

    best_trial_f1 = 0.0
    patience_count = 0

    for epoch in range(EPOCHS_PER_TRIAL):
        model.train()
        train_loss = 0

        # Barra de progreso
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False)

        for x_b, y_b in pbar:
            x_b, y_b = x_b.to(device, non_blocking=True), y_b.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # Forward + Loss
            with autocast():
                logits = model(x_b)
                loss = criterion(logits, y_b)

            # Backward
            scaler.scale(loss).backward()

            # Update
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            scaler.step(optimizer)
            scaler.update()

            # Acumular solo el valor escalar
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})

            # Borramos referencias
            del x_b, y_b, logits, loss

        # Limpieza al final de la época
        avg_train_loss = train_loss / len(train_loader)
        torch.cuda.empty_cache()

        # EVALUACIÓN
        model.eval()
        all_preds, all_targs = [], []

        with torch.no_grad():
            for x_b, y_b in eval_loader:
                x_b, y_b = x_b.to(device), y_b.to(device)

                with autocast():
                    logits = model(x_b)
                    preds = torch.softmax(logits[..., 1:5], dim=-1).argmax(dim=-1)

                targs = y_b[..., 1:5].argmax(dim=-1)
                mask = (y_b.sum(dim=-1) != 0)

                # Mover a CPU inmediatamente para liberar GPU
                all_preds.extend(preds[mask].cpu().numpy())
                all_targs.extend(targs[mask].cpu().numpy())

                # Limpiar batch de evaluación
                del x_b, y_b, logits, preds, targs, mask

        val_f1 = f1_score(all_targs, all_preds, average='macro', zero_division=0)

        print(f"   Ep {epoch+1}: Loss {avg_train_loss:.4f} | Val F1 (Punt): {val_f1:.4f}")

        # Lógica de guardado
        if val_f1 > best_trial_f1:
            best_trial_f1 = val_f1
            patience_count = 0
            if val_f1 > best_global_f1:
                print(f"      🏆 NUEVO RÉCORD: {val_f1:.4f}")
                best_global_f1 = val_f1
                best_global_params = params
                best_model_state = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print("      Early Stopping.")
                break

    # Limpieza final de Trial
    del model, optimizer
    torch.cuda.empty_cache()
    gc.collect()

print("\n" + "#"*50)
print(f"MEJOR F1 GLOBAL: {best_global_f1:.4f}")
print(f"MEJORES PARAMS: {best_global_params}")

if best_model_state:
    torch.save({
        'state': best_model_state,
        'params': best_global_params
    }, 'mejor_modelo_lstm_uni_amp.pth')
    print(" Modelo guardado.")

In [ ]:
best_params = {
    'hidden_size': 512,
    'num_layers': 3,
    'dropout': 0.3,
    'lr': 0.0005,
    'loss_weights': (1.0, 1.0, 1.0)
}

MODEL_PATH = 'mejor_modelo_lstm_uni_amp.pth'

model = NuestraLSTM_Uni(
    input_size=772,
    hidden_size=best_params['hidden_size'],
    num_layers=best_params['num_layers'],
    output_size=9,
    dropout_prob=best_params['dropout']
).to(device)

# Cargar los pesos guardados
checkpoint = torch.load(MODEL_PATH)
# A veces guardamos un diccionario, a veces solo el state_dict. Verificamos:
if 'state' in checkpoint:
    model.load_state_dict(checkpoint['state'])
elif 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint) # Si se guardó directo

model.eval()

# Listas para guardar todo
true_cap, pred_cap = [], []
true_ini, pred_ini = [], []
true_fin, pred_fin = [], []

with torch.no_grad():
    for x_b, y_b in tqdm(eval_loader, desc="Evaluando"):
        x_b = x_b.to(device)

        # Forward pass
        logits = model(x_b)

        # Puntuación Inicial (Binario: índice 0)
        # Usamos Sigmoid > 0.5 para decidir
        p_ini = (torch.sigmoid(logits[..., 0]) > 0.5).float()
        t_ini = y_b[..., 0]

        # Puntuación Final (Multiclase: índices 1-4)
        p_fin = torch.softmax(logits[..., 1:5], dim=-1).argmax(dim=-1)
        t_fin = y_b[..., 1:5].argmax(dim=-1)

        # Capitalización (Multiclase: índices 5-8)
        p_cap = torch.softmax(logits[..., 5:9], dim=-1).argmax(dim=-1)
        t_cap = y_b[..., 5:9].argmax(dim=-1)

        # Solo nos importan los tokens reales (donde el target no sea todo ceros)
        mask = (y_b.sum(dim=-1) != 0).cpu()

        # Guardamos en listas (CPU)
        true_ini.extend(t_ini.cpu()[mask].numpy())
        pred_ini.extend(p_ini.cpu()[mask].numpy())

        true_fin.extend(t_fin.cpu()[mask].numpy())
        pred_fin.extend(p_fin.cpu()[mask].numpy())

        true_cap.extend(t_cap.cpu()[mask].numpy())
        pred_cap.extend(p_cap.cpu()[mask].numpy())

# REPORTES

# Definimos los nombres de las clases para que se vea bien
nombres_fin = ['VACIO', 'PUNTO (.)', 'COMA (,)', 'SIGNO (?)']
nombres_cap = ['LOWER', 'TITLE (Mayus)', 'OTHER', 'UPPER']
nombres_ini = ['NO', 'SI (¿)']

print("\n" + "="*60)
print("RESULTADOS DETALLADOS - LSTM UNIDIRECCIONAL")
print("="*60)

print("\n--- 1. PUNTUACIÓN FINAL ---")
print(classification_report(true_fin, pred_fin, target_names=nombres_fin, digits=4))

print("\n--- 2. CAPITALIZACIÓN ---")
print(classification_report(true_cap, pred_cap, target_names=nombres_cap, digits=4))

print("\n--- 3. PUNTUACIÓN INICIAL (¿) ---")
print(classification_report(true_ini, pred_ini, target_names=nombres_ini, digits=4))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy
import gc
import os
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import f1_score

PARAMS = {
    'hidden_size': 512,
    'num_layers': 3,
    'dropout': 0.3,
    'lr': 0.0005,
    'loss_weights': (1.0, 1.0, 1.0)
}

# Rutas
MODEL_PATH_PREVIO = 'mejor_modelo_lstm_uni_amp.pth'
MODEL_PATH_NUEVO  = 'modelo_uni_final_extended.pth'

EXTRA_EPOCHS = 20
PATIENCE = 4
BATCH_SIZE = 32

print(f" Preparando Fine-Tuning con: {PARAMS}")


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = NuestraLSTM_Uni(
    input_size=772,
    hidden_size=PARAMS['hidden_size'],
    num_layers=PARAMS['num_layers'],
    output_size=9,
    dropout_prob=PARAMS['dropout']
).to(device)

print(" Cargando pesos previos...")
try:
    checkpoint = torch.load(MODEL_PATH_PREVIO)
    # Manejo de versiones de guardado
    if 'state' in checkpoint:
        model.load_state_dict(checkpoint['state'])
        best_f1 = checkpoint.get('f1', 0.59) # Recuperamos el score anterior
    elif 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        best_f1 = checkpoint.get('best_f1', 0.59)
    else:
        model.load_state_dict(checkpoint)
        best_f1 = 0.59 # Asumimos el valor que me dijiste

    print(f" Pesos cargados. Partimos de un F1 base de: {best_f1:.4f}")
except FileNotFoundError:
    print(" No encontré el archivo previo. Entrenaré desde cero.")
    best_f1 = 0.0

# OPTIMIZADOR Y SCHEDULER
criterion = MixedLoss(*PARAMS['loss_weights'])
optimizer = optim.Adam(model.parameters(), lr=PARAMS['lr'])
scaler = GradScaler()

# SCHEDULER: Si el F1 no mejora en 2 épocas, reduce el Learning Rate a la mitad
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

# BUCLE DE ENTRENAMIENTO EXTENDIDO
print(f"\n🏁 Iniciando entrenamiento extendido por {EXTRA_EPOCHS} épocas...")

best_model_state = copy.deepcopy(model.state_dict()) # Guardamos el inicial por si acaso
patience_counter = 0

for epoch in range(EXTRA_EPOCHS):
    model.train()
    train_loss = 0

    # Barra de progreso
    pbar = tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False)

    for x_b, y_b in pbar:
        x_b, y_b = x_b.to(device, non_blocking=True), y_b.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            logits = model(x_b)
            loss = criterion(logits, y_b)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})

        del x_b, y_b, logits, loss

    avg_train_loss = train_loss / len(train_loader)
    torch.cuda.empty_cache()

    # EVALUACIÓN
    model.eval()
    all_preds, all_targs = [], []

    with torch.no_grad():
        for x_b, y_b in eval_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)

            with autocast():
                logits = model(x_b)
                # Solo Puntuación Final para la métrica clave
                preds = torch.softmax(logits[..., 1:5], dim=-1).argmax(dim=-1)

            targs = y_b[..., 1:5].argmax(dim=-1)
            mask = (y_b.sum(dim=-1) != 0)

            all_preds.extend(preds[mask].cpu().numpy())
            all_targs.extend(targs[mask].cpu().numpy())

            del x_b, y_b, logits, preds, targs, mask

    val_f1 = f1_score(all_targs, all_preds, average='macro', zero_division=0)

    # Step del Scheduler (ajusta LR si es necesario)
    scheduler.step(val_f1)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"   Ep {epoch+1}: Loss {avg_train_loss:.4f} | Val F1: {val_f1:.4f} | LR: {current_lr:.1e}")

    # Guardado del mejor
    if val_f1 > best_f1:
        print(f"       MEJORA DETECTADA: {best_f1:.4f} -> {val_f1:.4f}")
        best_f1 = val_f1
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0

        # Guardado intermedio por seguridad
        torch.save({'state': best_model_state, 'params': PARAMS, 'f1': best_f1}, MODEL_PATH_NUEVO)
    else:
        patience_counter += 1
        print(f"       Sin mejora ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("       Early Stopping activado.")
            break

    gc.collect()

print("\n" + "="*50)
print(f"ENTRENAMIENTO FINALIZADO. MEJOR F1: {best_f1:.4f}")
print(f"Modelo guardado en: {MODEL_PATH_NUEVO}")


In [ ]:
import torch
import numpy as np
from sklearn.metrics import classification_report
from tqdm.auto import tqdm

#  CONFIGURACIÓN
MODEL_PATH = 'modelo_uni_final_extended.pth' # El archivo del Fine-Tuning
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Nombres de clases para que el reporte se entienda
# (Basado en la lógica de tu Dataset)
NOMBRES_FIN = ['VACIO', 'PUNTO (.)', 'COMA (,)', 'SIGNO (?)']
NOMBRES_CAP = ['MINUSCULA', 'TITLE (Mayus)', 'OTRO', 'MAYUSCULA']
NOMBRES_INI = ['NO', 'SI (¿)']

# CARGAR EL MODELO
print(f" Cargando modelo desde: {MODEL_PATH}")

# Reconstruimos la arquitectura con los mismos PARAMS que usamos
model = NuestraLSTM_Uni(
    input_size=772,
    hidden_size=PARAMS['hidden_size'],
    num_layers=PARAMS['num_layers'],
    output_size=9,
    dropout_prob=PARAMS['dropout']
).to(device)

checkpoint = torch.load(MODEL_PATH)
if 'state' in checkpoint:
    model.load_state_dict(checkpoint['state'])
else:
    model.load_state_dict(checkpoint)

model.eval()
print(" Modelo listo para evaluación.")

# GENERAR PREDICCIONES
print(" Procesando set de validación...")

true_fin, pred_fin = [], []
true_cap, pred_cap = [], []
true_ini, pred_ini = [], []

with torch.no_grad():
    for x_b, y_b in tqdm(eval_loader, desc="Evaluando"):
        x_b = x_b.to(device)

        # Forward
        logits = model(x_b)

        # PUNTUACIÓN FINAL (Indices 1-4)
        p_fin = torch.softmax(logits[..., 1:5], dim=-1).argmax(dim=-1)
        t_fin = y_b[..., 1:5].argmax(dim=-1)

        # CAPITALIZACIÓN (Indices 5-8)
        p_cap = torch.softmax(logits[..., 5:9], dim=-1).argmax(dim=-1)
        t_cap = y_b[..., 5:9].argmax(dim=-1)

        # PUNTUACIÓN INICIAL (Indice 0)
        # Binario: Sigmoid > 0.5
        p_ini = (torch.sigmoid(logits[..., 0]) > 0.5).float()
        t_ini = y_b[..., 0]

        # FILTRADO DE PADDING
        # Solo guardamos tokens reales (donde el target no sea todo ceros)
        mask = (y_b.sum(dim=-1) != 0).cpu()

        # Guardar en listas (CPU)
        true_fin.extend(t_fin.cpu()[mask].numpy())
        pred_fin.extend(p_fin.cpu()[mask].numpy())

        true_cap.extend(t_cap.cpu()[mask].numpy())
        pred_cap.extend(p_cap.cpu()[mask].numpy())

        true_ini.extend(t_ini.cpu()[mask].numpy())
        pred_ini.extend(p_ini.cpu()[mask].numpy())

# REPORTES FINALES

print("\n" + "="*60)
print(" REPORTE FINAL DE MÉTRICAS (LSTM UNIDIRECCIONAL)")
print("="*60)

print("\n TAREA 1: PUNTUACIÓN FINAL (La más difícil)")
print(classification_report(true_fin, pred_fin, target_names=NOMBRES_FIN, digits=4))

print("\n TAREA 2: CAPITALIZACIÓN")
print(classification_report(true_cap, pred_cap, target_names=NOMBRES_CAP, digits=4))

print("\n TAREA 3: PUNTUACIÓN INICIAL (Signos de apertura)")
print(classification_report(true_ini, pred_ini, target_names=NOMBRES_INI, digits=4))